In [1]:
!pip install faker

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 1.3 MB/s eta 0:00:02
   --------------- ------------------------ 0.8/2.0 MB 1.3 MB/s eta 0:00:01
   -------------------------- ------------- 1.3/2.0 MB 1.6 MB/s eta 0:00:01
   ------------------------------- -------- 1.6/2.0 MB 1.6 MB/s eta 0:00:01
   ------------------------------------ --- 1.8/2.0 MB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 1.5 MB/s  0:00:01


In [4]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)
fake = Faker()
Faker.seed(42)

print("Ready")

Ready


In [5]:
# Faker just generates realistic fake data
fake = Faker()
print(fake.company())       
print(fake.date_between(start_date=datetime(2023,1,1), end_date=datetime(2024,6,30)))

Rodriguez, Figueroa and Sanchez
2023-04-19


In [6]:
NUM_CUSTOMERS = 500

# The 3 types of customers — SMB = small business, Mid-Market = medium, Enterprise = large
segments = ["SMB", "Mid-Market", "Enterprise"]

# Empty list
customers = []

for i in range(NUM_CUSTOMERS):
    
    # Random signup date between Jan 2023 and Jun 2024
    signup = fake.date_between(
        start_date=datetime(2023, 1, 1),
        end_date=datetime(2024, 6, 30)
    )
    
    days_since_signup = (datetime(2024, 6, 30).date() - signup).days
    is_new = days_since_signup <= 90
    
    customers.append({
        "customer_id"    : f"CUST{str(i+1).zfill(4)}",  # CUST0001, CUST0002...
        "company_name"   : fake.company(),
        "segment"        : random.choices(segments, weights=[50, 30, 20])[0],
        "signup_date"    : signup,
        "is_new_customer": is_new
    })

df_customers = pd.DataFrame(customers)

# Always check your output
print(f"Total customers: {len(df_customers)}")
print(f"New customers  : {df_customers['is_new_customer'].sum()}")
print(f"\nSegment breakdown:")
print(df_customers['segment'].value_counts())
print(f"\nFirst 3 rows:")
df_customers.head(3)

Total customers: 500
New customers  : 85

Segment breakdown:
segment
SMB           248
Mid-Market    145
Enterprise    107
Name: count, dtype: int64

First 3 rows:


,customer_id,company_name,segment,signup_date,is_new_customer
0,CUST0001,"Yang, Gardner and Garza",Mid-Market,2023-03-21,False
1,CUST0002,Davis and Sons,SMB,2023-01-25,False
2,CUST0003,"Johnson, Gonzalez and Santos",SMB,2024-01-28,False


In [7]:
# The 3 plans and their monthly_prices
PLAN_PRICES = {
    "Basic"     : 2000,
    "Pro"       : 5000,
    "Enterprise": 12000
}

plans = list(PLAN_PRICES.keys())  # ["Basic", "Pro", "Enterprise"]

subscriptions = []

for _, row in df_customers.iterrows():
    
    if row["segment"] == "Enterprise":
        plan = random.choices(plans, weights=[10, 30, 60])[0]
    elif row["segment"] == "Mid-Market":
        plan = random.choices(plans, weights=[30, 50, 20])[0]
    else:  # SMB
        plan = random.choices(plans, weights=[60, 35, 5])[0]

    subscriptions.append({
        "subscription_id": f"SUB{row['customer_id'][4:]}",  # SUB0001, SUB0002...
        "customer_id"    : row["customer_id"],
        "plan_name"      : plan,
        "plan_price"     : PLAN_PRICES[plan],  # ground truth price
        "status"         : random.choices(
                               ["Active", "Cancelled", "Paused"],
                               weights=[80, 12, 8]
                           )[0]
    })

df_subscriptions = pd.DataFrame(subscriptions)

print(f"Total subscriptions: {len(df_subscriptions)}")
print(f"\nPlan breakdown:")
print(df_subscriptions["plan_name"].value_counts())
print(f"\nStatus breakdown:")
print(df_subscriptions["status"].value_counts())
print(f"\nFirst 3 rows:")
df_subscriptions.head(3)

Total subscriptions: 500

Plan breakdown:
plan_name
Pro           201
Basic         193
Enterprise    106
Name: count, dtype: int64

Status breakdown:
status
Active       404
Cancelled     61
Paused        35
Name: count, dtype: int64

First 3 rows:


,subscription_id,customer_id,plan_name,plan_price,status
0,SUB0001,CUST0001,Pro,5000,Active
1,SUB0002,CUST0002,Basic,2000,Active
2,SUB0003,CUST0003,Basic,2000,Active


In [8]:
invoices = []
invoice_counter = 1

df_merged = df_customers.merge(df_subscriptions, on="customer_id")

for _, row in df_merged.iterrows():
    
    if row["status"] == "Cancelled":
        continue
    
    # Generate one invoice per month from signup date to Jun 2024
    signup = pd.Timestamp(row["signup_date"])
    monthly_dates = pd.date_range(
        start=signup,
        end=pd.Timestamp("2024-06-30"),
        freq="MS"  # MS = Month Start — 1st of every month
    )
    
    for invoice_date in monthly_dates:
        
        # Due date is always 30 days after invoice is raised
        due_date = invoice_date + timedelta(days=30)
        
        if random.random() < 0.08:
            amount_billed = round(row["plan_price"] * random.uniform(0.4, 0.85), 2)
            has_billing_error = True
        else:
            amount_billed = float(row["plan_price"])
            has_billing_error = False
        
        today = datetime(2024, 6, 30)
        days_overdue = (today - due_date.to_pydatetime()).days
        
        if days_overdue < 0:
            payment_status = "Paid"
        else:
            payment_status = random.choices(
                ["Paid", "Unpaid", "Failed"],
                weights=[80, 15, 5]
            )[0]
        
        invoices.append({
            "invoice_id"       : f"INV{str(invoice_counter).zfill(5)}",
            "customer_id"      : row["customer_id"],
            "subscription_id"  : row["subscription_id"],
            "invoice_date"     : invoice_date.date(),
            "due_date"         : due_date.date(),
            "amount_billed"    : amount_billed,
            "plan_price"       : row["plan_price"],  
            "payment_status"   : payment_status,
            "has_billing_error": has_billing_error
        })
        invoice_counter += 1

df_invoices = pd.DataFrame(invoices)

print(f"Total invoices     : {len(df_invoices)}")
print(f"\nPayment status breakdown:")
print(df_invoices["payment_status"].value_counts())
print(f"\nBilling errors     : {df_invoices['has_billing_error'].sum()}")
print(f"\nFirst 3 rows:")
df_invoices.head(3)

Total invoices     : 3656

Payment status breakdown:
payment_status
Paid      3052
Unpaid     433
Failed     171
Name: count, dtype: int64

Billing errors     : 271

First 3 rows:


,invoice_id,customer_id,subscription_id,invoice_date,due_date,amount_billed,plan_price,payment_status,has_billing_error
0,INV00001,CUST0001,SUB0001,2023-04-01,2023-05-01,5000.0,5000,Failed,False
1,INV00002,CUST0001,SUB0001,2023-05-01,2023-05-31,5000.0,5000,Paid,False
2,INV00003,CUST0001,SUB0001,2023-06-01,2023-07-01,5000.0,5000,Paid,False


In [9]:
payments = []
payment_counter = 1

paid_invoices = df_invoices[df_invoices["payment_status"] == "Paid"]

for _, inv in paid_invoices.iterrows():
    
    payment_date = pd.Timestamp(inv["due_date"]) - timedelta(days=random.randint(0, 25))
    
    payments.append({
        "payment_id"  : f"PAY{str(payment_counter).zfill(5)}",
        "invoice_id"  : inv["invoice_id"],
        "customer_id" : inv["customer_id"],
        "payment_date": payment_date.date(),
        "amount_paid" : inv["amount_billed"]  
    })
    payment_counter += 1

df_payments = pd.DataFrame(payments)

print(f"Total payments: {len(df_payments)}")
print(f"\nFirst 3 rows:")
df_payments.head(3)

Total payments: 3052

First 3 rows:


,payment_id,invoice_id,customer_id,payment_date,amount_paid
0,PAY00001,INV00002,CUST0001,2023-05-10,5000.00
1,PAY00002,INV00003,CUST0001,2023-06-14,5000.00
2,PAY00003,INV00004,CUST0001,2023-07-15,3751.66


In [10]:
discounts = []
discount_counter = 1

promo_codes = {
    "NEWCUST20": {"pct": 0.20, "new_only": True},   # 20% off — new customers only
    "PROMO10"  : {"pct": 0.10, "new_only": False},  # 10% off — everyone eligible
    "SAVE15"   : {"pct": 0.15, "new_only": False},  # 15% off — everyone eligible
}

# Apply discounts to 25% of invoices randomly
sampled_invoices = df_invoices.sample(frac=0.25, random_state=42)

for _, inv in sampled_invoices.iterrows():
    
    code = random.choice(list(promo_codes.keys()))
    promo = promo_codes[code]
    
    discount_amt = round(inv["amount_billed"] * promo["pct"], 2)
    
    is_new = df_customers.loc[
        df_customers["customer_id"] == inv["customer_id"],
        "is_new_customer"
    ].values[0]
    
    if promo["new_only"] and not is_new:
        is_eligible = False  
    else:
        is_eligible = True
    
    discounts.append({
        "discount_id"    : f"DIS{str(discount_counter).zfill(5)}",
        "invoice_id"     : inv["invoice_id"],
        "customer_id"    : inv["customer_id"],
        "discount_code"  : code,
        "discount_amount": discount_amt,
        "is_eligible"    : is_eligible
    })
    discount_counter += 1

df_discounts = pd.DataFrame(discounts)

print(f"Total discounts    : {len(df_discounts)}")
print(f"Ineligible discounts (Leak 2): {len(df_discounts[df_discounts['is_eligible']==False])}")
print(f"\nDiscount code breakdown:")
print(df_discounts["discount_code"].value_counts())
print(f"\nFirst 3 rows:")
df_discounts.head(3)

Total discounts    : 914
Ineligible discounts (Leak 2): 280

Discount code breakdown:
discount_code
PROMO10      323
SAVE15       303
NEWCUST20    288
Name: count, dtype: int64

First 3 rows:


,discount_id,invoice_id,customer_id,discount_code,discount_amount,is_eligible
0,DIS00001,INV00496,CUST0063,SAVE15,750.0,True
1,DIS00002,INV01133,CUST0138,PROMO10,500.0,True
2,DIS00003,INV01002,CUST0122,NEWCUST20,1000.0,True


In [11]:
# Saves all tables to one Excel file - one sheet per table
with pd.ExcelWriter("saas_revenue_leak_raw.xlsx", engine="openpyxl") as writer:
    df_customers.to_excel(writer,      sheet_name="customers",      index=False)
    df_subscriptions.to_excel(writer,  sheet_name="subscriptions",  index=False)
    df_invoices.to_excel(writer,       sheet_name="invoices",        index=False)
    df_payments.to_excel(writer,       sheet_name="payments",        index=False)
    df_discounts.to_excel(writer,      sheet_name="discounts",       index=False)

print("Saved → saas_revenue_leak_raw.xlsx")
print("\nSheets inside the file:")
print("  • customers")
print("  • subscriptions")
print("  • invoices")
print("  • payments")
print("  • discounts")

Saved → saas_revenue_leak_raw.xlsx

Sheets inside the file:
  • customers
  • subscriptions
  • invoices
  • payments
  • discounts


In [12]:
import sqlite3

# create database file
conn = sqlite3.connect("saas_revenue_leak.db")
cursor = conn.cursor()

print("Database created ✓")
print("Connected to: saas_revenue_leak.db")

Database created ✓
Connected to: saas_revenue_leak.db


In [13]:
# load all 5 dataframes into SQLite database
df_customers.to_sql("customers",         conn, if_exists="replace", index=False)
df_subscriptions.to_sql("subscriptions", conn, if_exists="replace", index=False)
df_invoices.to_sql("invoices",           conn, if_exists="replace", index=False)
df_payments.to_sql("payments",           conn, if_exists="replace", index=False)
df_discounts.to_sql("discounts",         conn, if_exists="replace", index=False)

print("All 5 tables loaded into database ✓")
print("\nTables in database:")

tables = cursor.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
for table in tables:
    print(f"  • {table[0]}")

All 5 tables loaded into database ✓

Tables in database:
  • customers
  • subscriptions
  • invoices
  • payments
  • discounts


In [14]:
leak1 = pd.read_sql_query("""
    SELECT 
        i.customer_id,
        c.company_name,
        c.segment,
        i.invoice_id,
        i.invoice_date,
        i.due_date,
        i.amount_billed,
        i.payment_status,
        -- How many days has this invoice been overdue?
        CAST(julianday('2024-06-30') - julianday(i.due_date) AS INT) AS days_overdue,
        -- Bucket into 30/60/90+ day categories
        CASE 
            WHEN CAST(julianday('2024-06-30') - julianday(i.due_date) AS INT) BETWEEN 1  AND 30 THEN '1-30 days'
            WHEN CAST(julianday('2024-06-30') - julianday(i.due_date) AS INT) BETWEEN 31 AND 60 THEN '31-60 days'
            WHEN CAST(julianday('2024-06-30') - julianday(i.due_date) AS INT) BETWEEN 61 AND 90 THEN '61-90 days'
            ELSE '90+ days'
        END AS overdue_bucket
    FROM invoices i
    JOIN customers c ON i.customer_id = c.customer_id
    WHERE i.payment_status IN ('Unpaid', 'Failed')
    ORDER BY days_overdue DESC
""", conn)

print(f"Total overdue invoices : {len(leak1)}")
print(f"Total rupee value      : ₹{leak1['amount_billed'].sum():,.2f}")
print(f"\nBreakdown by bucket:")
print(leak1.groupby('overdue_bucket')['amount_billed'].agg(['count','sum']).rename(columns={'count':'invoices','sum':'amount'}))
print(f"\nTop 5 worst offenders:")
leak1.groupby('company_name')['amount_billed'].sum().sort_values(ascending=False).head()

Total overdue invoices : 604
Total rupee value      : ₹3,214,130.33

Breakdown by bucket:
                invoices      amount
overdue_bucket                      
1-30 days             83   410645.19
31-60 days            61   340553.77
90+ days             460  2462931.37

Top 5 worst offenders:


company_name
Diaz, Anderson and Browning    89387.22
Gaines, Harrell and Evans      57860.66
Rivera-Kennedy                 48000.00
Obrien-Dixon                   48000.00
Stanley LLC                    48000.00
Name: amount_billed, dtype: float64

In [15]:
leak2 = pd.read_sql_query("""
    SELECT
        d.customer_id,
        c.company_name,
        c.segment,
        c.is_new_customer,
        d.invoice_id,
        d.discount_code,
        d.discount_amount,
        d.is_eligible
    FROM discounts d
    JOIN customers c ON d.customer_id = c.customer_id
    WHERE d.is_eligible = 0
    ORDER BY d.discount_amount DESC
""", conn)

print(f"Total ineligible discounts : {len(leak2)}")
print(f"Total rupee value          : ₹{leak2['discount_amount'].sum():,.2f}")
print(f"\nBreakdown by discount code:")
print(leak2.groupby('discount_code')['discount_amount'].agg(['count','sum']).rename(columns={'count':'occurrences','sum':'amount'}))
print(f"\nTop 5 customers abusing discounts:")
leak2.groupby('company_name')['discount_amount'].sum().sort_values(ascending=False).head()

Total ineligible discounts : 280
Total rupee value          : ₹285,858.63

Breakdown by discount code:
               occurrences     amount
discount_code                        
NEWCUST20              280  285858.63

Top 5 customers abusing discounts:


company_name
Holmes LLC                   7200.00
Williams, Smith and Morse    7200.00
Mahoney Inc                  7200.00
Carlson LLC                  6720.74
Barr-Peterson                4800.00
Name: discount_amount, dtype: float64

In [16]:
leak3 = pd.read_sql_query("""
    SELECT
        i.customer_id,
        c.company_name,
        c.segment,
        s.plan_name,
        i.invoice_id,
        i.invoice_date,
        i.plan_price,
        i.amount_billed,
        -- The gap between what should have been billed vs what was billed
        ROUND(i.plan_price - i.amount_billed, 2) AS amount_undercharged,
        i.payment_status
    FROM invoices i
    JOIN customers c ON i.customer_id = c.customer_id
    JOIN subscriptions s ON i.subscription_id = s.subscription_id
    WHERE i.has_billing_error = 1
    ORDER BY amount_undercharged DESC
""", conn)

print(f"Total billing errors  : {len(leak3)}")
print(f"Total rupee value     : ₹{leak3['amount_undercharged'].sum():,.2f}")
print(f"\nBreakdown by plan:")
print(leak3.groupby('plan_name')['amount_undercharged'].agg(['count','sum']).rename(columns={'count':'errors','sum':'amount'}))
print(f"\nTop 5 most undercharged customers:")
leak3.groupby('company_name')['amount_undercharged'].sum().sort_values(ascending=False).head()

Total billing errors  : 271
Total rupee value     : ₹516,654.11

Breakdown by plan:
            errors     amount
plan_name                    
Basic          111   84878.53
Enterprise      54  239689.76
Pro            106  192085.82

Top 5 most undercharged customers:


company_name
Garcia-Nolan               15243.67
Smith LLC                  11690.41
Hays-Daniels               10383.33
Daniels, Vance and York    10369.02
Brown-Stone                 9944.58
Name: amount_undercharged, dtype: float64

In [17]:
leak_summary = pd.read_sql_query("""
    SELECT
        c.customer_id,
        c.company_name,
        c.segment,
        s.plan_name,
        
        -- Leak 1: unpaid/failed invoices
        COALESCE(l1.unpaid_amount, 0) AS leak1_overdue,
        
        -- Leak 2: ineligible discounts
        COALESCE(l2.discount_abuse, 0) AS leak2_discount_abuse,
        
        -- Leak 3: billing errors
        COALESCE(l3.billing_error, 0) AS leak3_billing_error,
        
        -- Total leak per customer
        ROUND(
            COALESCE(l1.unpaid_amount, 0) + 
            COALESCE(l2.discount_abuse, 0) + 
            COALESCE(l3.billing_error, 0), 2
        ) AS total_leak_amount

    FROM customers c
    JOIN subscriptions s ON c.customer_id = s.customer_id

    -- Leak 1 subquery
    LEFT JOIN (
        SELECT customer_id, SUM(amount_billed) AS unpaid_amount
        FROM invoices
        WHERE payment_status IN ('Unpaid', 'Failed')
        GROUP BY customer_id
    ) l1 ON c.customer_id = l1.customer_id

    -- Leak 2 subquery
    LEFT JOIN (
        SELECT customer_id, SUM(discount_amount) AS discount_abuse
        FROM discounts
        WHERE is_eligible = 0
        GROUP BY customer_id
    ) l2 ON c.customer_id = l2.customer_id

    -- Leak 3 subquery
    LEFT JOIN (
        SELECT customer_id, SUM(plan_price - amount_billed) AS billing_error
        FROM invoices
        WHERE has_billing_error = 1
        GROUP BY customer_id
    ) l3 ON c.customer_id = l3.customer_id

    WHERE 
        COALESCE(l1.unpaid_amount, 0) + 
        COALESCE(l2.discount_abuse, 0) + 
        COALESCE(l3.billing_error, 0) > 0

    ORDER BY total_leak_amount DESC
""", conn)

print(f"Customers with at least one leak : {len(leak_summary)}")
print(f"\n=== TOTAL REVENUE AT RISK ===")
print(f"Leak 1 — Overdue payments  : ₹{leak_summary['leak1_overdue'].sum():>12,.2f}")
print(f"Leak 2 — Discount abuse    : ₹{leak_summary['leak2_discount_abuse'].sum():>12,.2f}")
print(f"Leak 3 — Billing errors    : ₹{leak_summary['leak3_billing_error'].sum():>12,.2f}")
print(f"{'─'*45}")
print(f"TOTAL REVENUE LEAK         : ₹{leak_summary['total_leak_amount'].sum():>12,.2f}")
print(f"\nTop 10 customers by total leak:")
leak_summary[['company_name','segment','plan_name',
              'leak1_overdue','leak2_discount_abuse',
              'leak3_billing_error','total_leak_amount']].head(10)

Customers with at least one leak : 346

=== TOTAL REVENUE AT RISK ===
Leak 1 — Overdue payments  : ₹3,214,130.33
Leak 2 — Discount abuse    : ₹  285,858.63
Leak 3 — Billing errors    : ₹  516,654.11
─────────────────────────────────────────────
TOTAL REVENUE LEAK         : ₹4,016,643.07

Top 10 customers by total leak:


,company_name,segment,plan_name,leak1_overdue,leak2_discount_abuse,leak3_billing_error,total_leak_amount
0,"Diaz, Anderson and Browning",Enterprise,Enterprise,89387.22,0.0,6612.78,96000.00
1,"Gaines, Harrell and Evans",Enterprise,Enterprise,57860.66,0.0,2139.34,60000.00
2,Maxwell-Jones,Enterprise,Enterprise,48000.00,4800.0,6339.46,59139.46
3,Garcia-Nolan,Enterprise,Enterprise,36000.00,2400.0,15243.67,53643.67
4,Stanley LLC,SMB,Enterprise,48000.00,2400.0,0.00,50400.00
5,Rivera-Kennedy,SMB,Enterprise,48000.00,2400.0,0.00,50400.00
6,Holmes LLC,Enterprise,Enterprise,36000.00,7200.0,7174.81,50374.81
7,Obrien-Dixon,Enterprise,Enterprise,48000.00,0.0,0.00,48000.00
8,Gay Inc,Enterprise,Enterprise,48000.00,0.0,0.00,48000.00
9,"Martinez, Richardson and Curry",Enterprise,Enterprise,36000.00,0.0,6597.63,42597.63


In [18]:
with pd.ExcelWriter("saas_revenue_leak_analysis.xlsx", engine="openpyxl") as writer:
    leak1.to_excel(writer,        sheet_name="Leak1_Overdue",        index=False)
    leak2.to_excel(writer,        sheet_name="Leak2_Discount_Abuse", index=False)
    leak3.to_excel(writer,        sheet_name="Leak3_Billing_Errors", index=False)
    leak_summary.to_excel(writer, sheet_name="Master_Leak_Summary",  index=False)

print("Saved → saas_revenue_leak_analysis.xlsx")
print("\nSheets:")
print("  • Leak1_Overdue")
print("  • Leak2_Discount_Abuse")
print("  • Leak3_Billing_Errors")
print("  • Master_Leak_Summary")

Saved → saas_revenue_leak_analysis.xlsx

Sheets:
  • Leak1_Overdue
  • Leak2_Discount_Abuse
  • Leak3_Billing_Errors
  • Master_Leak_Summary
